# MCD-rPPG Dataset: Inference Visualization Showcase

## Overview
This notebook demonstrates the inference pipeline for the MCD-rPPG dataset:

1. **Video Selection**: Choose a video from the dataset for inference
2. **Chunking**: Split video into 450-frame chunks with 150-frame overlap
3. **ROI Extraction**: Extract 9 facial regions using MediaPipe landmarks
4. **Visualization**: Plot first, middle, and last frames with ROI boxes
5. **Individual ROI Display**: Show extracted ROIs for each frame position
6. **Shape Information**: Display dimensions for inference compatibility

### Key Parameters:
- **Chunk Size**: 450 frames
- **Overlap**: 150 frames
- **ROIs**: 9 regions (full_face, forehead, left_eye, right_eye, nose, mouth, chin, right_cheek_50, left_cheek_280, chin_199)
- **ROI Box Size**: 24x24 pixels
- **Output**: ROI tensors ready for model inference

### Usage:
This notebook is designed to validate that preprocessing produces correctly formatted data for inference models.

In [ ]:
# Install required packages
!pip install imageio[ffmpeg] -q
!pip install mediapipe -q
!pip install scikit-video -q
!pip install opencv-python -q
!pip install numba -q
!pip install scipy -q

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import imageio
import cv2
from IPython.display import display
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')

# Configuration
DATASET_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/'
DB_PATH = os.path.join(DATASET_PATH, 'db.csv')
MEDIAPIPE_MODEL_PATH = '/home/cristic/face_landmarker.task'
OUTPUT_PATH = '/home/cristic/preprocessed_data'

# Chunking parameters
CHUNK_SIZE = 450
OVERLAP_SIZE = 150

# ROI Configuration (Corrected Canonical MediaPipe Mesh Indices)
ROIS = {
    'full_face': list(range(468)),
    
    # Corrected canonical feature clusters
    'forehead': [10, 67, 69, 108, 109, 151, 337, 338, 297, 299, 9, 8],
    'left_eye': [33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246],
    'right_eye': [362, 382, 381, 380, 374, 373, 390, 249, 263, 466, 388, 387, 386, 385, 384, 398],
    'nose': [1, 2, 98, 327, 328, 2, 4, 5, 195, 197, 6, 168],
    'mouth': [61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291, 308, 324, 318, 402, 317, 14, 87, 178, 95],
    'chin': [152, 148, 176, 149, 150, 136, 172, 377, 400, 378, 379, 365, 397],
    
    # Specific landmark targets
    'right_cheek_50': [50],
    'left_cheek_280': [280],
    'chin_199': [199]
}

ROI_BOX_SIZE = (24, 24)
SMOOTHING_WINDOW = 5

print('Configuration loaded successfully')
print(f'Dataset path: {DATASET_PATH}')
print(f'ROIs available: {list(ROIS.keys())}')
print(f'ROI box size: {ROI_BOX_SIZE}')

## 1. Video Selection
Select a video from the dataset for inference visualization.

In [ ]:
# Load database
df = pd.read_csv(DB_PATH)
meta_df = df.copy()

file_columns = ['ecg', 'video', 'meta', 'ppg_sync']
for col in file_columns:
    if col in meta_df.columns:
        meta_df[f'{col}_full'] = meta_df[col].apply(
            lambda x: os.path.join(DATASET_PATH, x) if not os.path.isabs(x) else x
        )

meta_df['subject_id'] = meta_df['patient_id']
meta_df['condition'] = meta_df['step']
meta_df['camera_type'] = meta_df['camera']
meta_df['view_type'] = meta_df['view']

print(f'Total videos in dataset: {len(meta_df)}')
print(f'Camera types: {meta_df["camera_type"].unique()}')
print(f'Conditions: {meta_df["condition"].unique()}')
print(f'Subjects: {meta_df["subject_id"].nunique()}')

# Show a sample of available videos
sample_videos = meta_df.dropna(subset=['video_full', 'ppg_sync_full']).head(10)
print('\nSample videos:')
display(sample_videos[['subject_id', 'camera_type', 'condition', 'view_type']].head(10))

In [ ]:
# Select the first available video for demonstration
# You can change this to select a different video
available_videos = meta_df.dropna(subset=['video_full', 'ppg_sync_full'])

# Filter to get a specific video - let's use the first one with all data
selected_video = available_videos.iloc[0]

print('Selected Video for Inference:')
print(f'  Subject ID: {selected_video["subject_id"]}')
print(f'  Camera Type: {selected_video["camera_type"]}')
print(f'  Condition: {selected_video["condition"]}')
print(f'  View Type: {selected_video["view_type"]}')
print(f'  Video Path: {selected_video["video_full"]}')
print(f'  PPG Sync Path: {selected_video["ppg_sync_full"]}')

# Store paths for processing
VIDEO_PATH = selected_video['video_full']
PPG_SYNC_PATH = selected_video['ppg_sync_full']
META_DATA = {
    'subject_id': selected_video['subject_id'],
    'condition': selected_video['condition'],
    'camera_type': selected_video['camera_type'],
    'view_type': selected_video['view_type'],
}

print('Video selected successfully!')

## 2. MediaPipe Landmark Detection Setup

In [ ]:
try:
    import mediapipe as mp
    from mediapipe.tasks import python
    from mediapipe.tasks.python import vision
    MEDIAPIPE_AVAILABLE = True
    print('MediaPipe available')
except ImportError as e:
    MEDIAPIPE_AVAILABLE = False
    print(f'MediaPipe not available: {e}')

class TemporalSmoother:
    def __init__(self, window_size=5):
        self.window_size = window_size
        self.history = []

    def smooth(self, landmarks):
        self.history.append(landmarks.copy())
        if len(self.history) > self.window_size:
            self.history.pop(0)
        n = len(self.history)
        weights = [float(i + 1) for i in range(n)]
        smoothed = np.zeros_like(landmarks)
        for i, w in enumerate(weights):
            smoothed += self.history[i] * w
        smoothed /= sum(weights)
        return smoothed

class MediaPipeLandmarkDetector:
    def __init__(self, model_path, smoothing_window=5):
        self.model_path = model_path
        self.smoothing_window = smoothing_window
        self.smoother = TemporalSmoother(smoothing_window)
        self.detector = None
        self.frame_count = 0
        self.fps = 30.0

    def initialize_detector(self):
        if not MEDIAPIPE_AVAILABLE:
            raise RuntimeError('MediaPipe not available')
        base_options = python.BaseOptions(model_asset_path=self.model_path)
        options = vision.FaceLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.VIDEO,
            output_face_blendshapes=True,
            output_facial_transformation_matrixes=True,
            num_faces=1
        )
        self.detector = vision.FaceLandmarker.create_from_options(options)

    def detect_landmarks(self, frame):
        if self.detector is None:
            self.initialize_detector()
        if frame.dtype != np.uint8:
            frame = (frame * 255).astype(np.uint8)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
        try:
            timestamp_ms = int(self.frame_count * (1000.0 / self.fps))
            result = self.detector.detect_for_video(mp_image, timestamp_ms)
            self.frame_count += 1
            if result and result.face_landmarks:
                face_landmarks = result.face_landmarks[0]
                frame_width, frame_height = frame.shape[1], frame.shape[0]
                points = np.array([(lm.x * frame_width, lm.y * frame_height) for lm in face_landmarks], dtype='float32')
                if np.any(np.isnan(points)) or np.any(np.isinf(points)):
                    return None
                if np.max(points) > max(frame_width, frame_height) * 3 or np.min(points) < -max(frame_width, frame_height) * 2:
                    return None
                smoothed_points = self.smoother.smooth(points)
                return smoothed_points
            else:
                return None
        except Exception as e:
            print(f'Error in landmark detection: {e}')
            return None

    def reset(self):
        self.frame_count = 0
        self.smoother.history = []
        if self.detector is not None:
            try:
                self.detector.close()
            except Exception as e:
                print(f'Warning during detector close: {e}')
            self.detector = None
        self.initialize_detector()

if MEDIAPIPE_AVAILABLE:
    detector = MediaPipeLandmarkDetector(MEDIAPIPE_MODEL_PATH, smoothing_window=SMOOTHING_WINDOW)
    print(f'MediaPipe detector initialized with smoothing window = {SMOOTHING_WINDOW}')
else:
    print('MediaPipe not available - landmark detection will be skipped')

## 3. Chunking Strategy for Inference

In [ ]:
def count_video_frames_ffmpeg(video_path):
    try:
        reader = imageio.get_reader(video_path, 'ffmpeg')
        n_frames = reader.count_frames()
        reader.close()
        return n_frames
    except Exception as e:
        print(f'Error: {e}')
        return 0

def get_video_fps(video_path):
    try:
        reader = imageio.get_reader(video_path, 'ffmpeg')
        fps = reader.get_meta_data().get('fps', 30.0)
        reader.close()
        return fps
    except Exception:
        return 30.0

def create_chunks(n_frames, chunk_size=450, overlap_size=150):
    chunks = []
    chunk_idx = 0
    start = 0
    while start < n_frames:
        end = min(start + chunk_size, n_frames)
        chunks.append((start, end, chunk_idx))
        if end == n_frames:
            break
        start = end - overlap_size
        chunk_idx += 1
    return chunks

def load_video_frames(video_path, start_frame=0, end_frame=None):
    try:
        reader = imageio.get_reader(video_path, 'ffmpeg')
        total_frames = reader.count_frames()
        if end_frame is None:
            end_frame = total_frames
        frames = []
        for i in range(start_frame, end_frame):
            frames.append(reader.get_next_data())
        reader.close()
        return np.array(frames)
    except Exception as e:
        print(f'Error: {e}')
        return np.array([])

# Count frames in selected video
n_frames = count_video_frames_ffmpeg(VIDEO_PATH)
fps = get_video_fps(VIDEO_PATH)
print(f'Video: {VIDEO_PATH}')
print(f'Total frames: {n_frames}')
print(f'FPS: {fps:.2f}')
print(f'Duration: {n_frames / fps:.2f} seconds')

# Create chunks
chunks = create_chunks(n_frames, CHUNK_SIZE, OVERLAP_SIZE)
print(f'\nChunking with size={CHUNK_SIZE}, overlap={OVERLAP_SIZE}:')
print(f'Total chunks: {len(chunks)}')
for start, end, idx in chunks:
    print(f'  Chunk {idx}: frames {start}-{end} ({end-start} frames)')

## 4. ROI Extraction Functions

In [ ]:
def extract_roi_bbox(landmarks, roi_indices, frame_shape, box_size=(24, 24)):
    valid_indices = [i for i in roi_indices if i < len(landmarks)]
    if not valid_indices:
        return (0, 0, box_size[0], box_size[1])
    roi_points = landmarks[valid_indices]
    raw_x = np.mean(roi_points[:, 0])
    raw_y = np.mean(roi_points[:, 1])
    if not np.isfinite(raw_x) or not np.isfinite(raw_y):
        return (0, 0, box_size[0], box_size[1])
    center_x = max(0, min(int(raw_x), frame_shape[1]))
    center_y = max(0, min(int(raw_y), frame_shape[0]))
    box_w, box_h = box_size
    x = max(0, center_x - box_w // 2)
    y = max(0, center_y - box_h // 2)
    x = min(x, frame_shape[1] - box_w)
    y = min(y, frame_shape[0] - box_h)
    return (int(x), int(y), int(box_w), int(box_h))

def extract_roi_region(frame, bbox):
    x, y, w, h = bbox
    return frame[y:y+h, x:x+w]

def extract_all_rois_for_frame(frame, landmarks, rois_dict, box_size=(24, 24)):
    frame_shape = frame.shape[:2]
    roi_data = {}
    for roi_name, roi_indices in rois_dict.items():
        bbox = extract_roi_bbox(landmarks, roi_indices, frame_shape, box_size)
        roi_region = extract_roi_region(frame, bbox)
        roi_data[roi_name] = roi_region
    return roi_data

print('ROI extraction functions ready')

## 5. Process First Chunk for Inference Visualization
Process the first chunk to extract ROIs and prepare for inference visualization.

In [ ]:
if MEDIAPIPE_AVAILABLE:
    print('Processing first chunk for inference visualization...')
    
    # Get first chunk
    if chunks:
        start, end, chunk_idx = chunks[0]
        print(f'Processing chunk {chunk_idx}: frames {start}-{end}')
        
        # Load frames for this chunk
        chunk_frames = load_video_frames(VIDEO_PATH, start, end)
        print(f'Loaded {len(chunk_frames)} frames')
        
        # Detect landmarks for all frames
        detector.reset()
        detector.fps = fps
        
        chunk_landmarks = []
        for frame in tqdm(chunk_frames, desc='Detecting landmarks'):
            lms = detector.detect_landmarks(frame)
            if lms is not None:
                chunk_landmarks.append(lms)
            elif chunk_landmarks:
                chunk_landmarks.append(chunk_landmarks[-1].copy())
            else:
                chunk_landmarks.append(np.zeros((468, 2), dtype='float32'))
        
        chunk_landmarks = np.array(chunk_landmarks)
        print(f'Landmarks shape: {chunk_landmarks.shape}')
        
        # Extract ROIs for all frames
        roi_data = {}
        for roi_name, roi_indices in ROIS.items():
            roi_frames = []
            for frame_idx, frame in enumerate(chunk_frames):
                landmarks = chunk_landmarks[frame_idx]
                if np.all(landmarks == 0):
                    h, w = frame.shape[:2]
                    bbox = (w // 2 - ROI_BOX_SIZE[0] // 2, h // 2 - ROI_BOX_SIZE[1] // 2, ROI_BOX_SIZE[0], ROI_BOX_SIZE[1])
                else:
                    bbox = extract_roi_bbox(landmarks, roi_indices, frame.shape[:2], ROI_BOX_SIZE)
                roi_region = extract_roi_region(frame, bbox)
                roi_frames.append(roi_region)
            roi_data[roi_name] = np.array(roi_frames)
        
        print(f'ROI data extracted for {len(roi_data)} regions')
        
        # Store for visualization
        inference_data = {
            'frames': chunk_frames,
            'landmarks': chunk_landmarks,
            'roi_data': roi_data,
            'start_frame': start,
            'end_frame': end,
            'chunk_idx': chunk_idx,
            'metadata': META_DATA
        }
        print('Inference data ready for visualization!')
    else:
        print('No chunks available')
else:
    print('MediaPipe not available - skipping processing')

## 6. Visualization: Frames with ROI Boxes
Plot first, middle, and last frames with ROI bounding boxes overlaid.

In [ ]:
if MEDIAPIPE_AVAILABLE and 'inference_data' in locals():
    print('=' * 80)
    print('VISUALIZATION: Frames with ROI Bounding Boxes')
    print('=' * 80)
    
    frames = inference_data['frames']
    landmarks_list = inference_data['landmarks']
    n_frames = len(frames)
    start_frame = inference_data['start_frame']
    
    # Select first, middle, and last frames
    frame_indices = [0, n_frames // 2, n_frames - 1]
    frame_labels = [f'Frame {start_frame + idx}' for idx in frame_indices]
    
    # Create figure
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    for ax_idx, (frame_idx, label) in enumerate(zip(frame_indices, frame_labels)):
        frame = frames[frame_idx]
        landmarks = landmarks_list[frame_idx]
        
        # Display frame
        axes[ax_idx].imshow(frame)
        axes[ax_idx].set_title(label, fontsize=12, fontweight='bold')
        
        # Draw ROI boxes for all regions except full_face
        for roi_name, roi_indices in ROIS.items():
            if roi_name == 'full_face':
                continue
            if np.all(landmarks == 0):
                h, w = frame.shape[:2]
                bbox = (w // 2 - ROI_BOX_SIZE[0] // 2, h // 2 - ROI_BOX_SIZE[1] // 2, ROI_BOX_SIZE[0], ROI_BOX_SIZE[1])
            else:
                bbox = extract_roi_bbox(landmarks, roi_indices, frame.shape[:2], ROI_BOX_SIZE)
            x, y, box_w, box_h = bbox
            
            # Different colors for different ROI types
            if 'eye' in roi_name:
                color = 'cyan'
            elif 'forehead' in roi_name:
                color = 'magenta'
            elif 'nose' in roi_name:
                color = 'green'
            elif 'mouth' in roi_name:
                color = 'red'
            elif 'chin' in roi_name:
                color = 'blue'
            elif 'cheek' in roi_name:
                color = 'orange'
            else:
                color = 'yellow'
            
            # Draw rectangle
            rect = patches.Rectangle(
                (x, y), box_w, box_h,
                linewidth=1.5, edgecolor=color, facecolor='none', linestyle='-'
            )
            axes[ax_idx].add_patch(rect)
            
            # Add label
            axes[ax_idx].text(x, y - 3, roi_name,
                           color=color, fontsize=7,
                           bbox=dict(facecolor=color, alpha=0.6, edgecolor='none', pad=1))
        
        axes[ax_idx].axis('off')
    
    plt.suptitle(
        f'ROI Bounding Boxes - {META_DATA["camera_type"]} Camera | ' \
        f'Subject: {META_DATA["subject_id"]} | Condition: {META_DATA["condition"]}',
        fontsize=14, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.show()
    
    print(f'Displayed frames: {[start_frame + idx for idx in frame_indices]}')
    print(f'ROI box size: {ROI_BOX_SIZE}')
else:
    print('No inference data available for visualization')

## 7. Individual ROI Extraction Visualization
Display extracted ROI regions for first, middle, and last frames.

In [ ]:
if MEDIAPIPE_AVAILABLE and 'inference_data' in locals():
    print('=' * 80)
    print('VISUALIZATION: Individual ROI Extractions')
    print('=' * 80)
    
    roi_data = inference_data['roi_data']
    frames = inference_data['frames']
    n_frames = len(frames)
    start_frame = inference_data['start_frame']
    
    # Select first, middle, and last frames
    frame_indices = [0, n_frames // 2, n_frames - 1]
    
    # Filter ROIs to show (exclude full_face as it's too large)
    roi_names = [name for name in ROIS.keys() if name != 'full_face']
    n_rois = len(roi_names)
    
    # Create grid: 3 rows (frames) x N columns (ROIs)
    fig, axes = plt.subplots(3, n_rois, figsize=(18, 6))
    
    if n_rois == 1:
        axes = axes.reshape(3, 1)
    
    for row_idx, frame_idx in enumerate(frame_indices):
        for col_idx, roi_name in enumerate(roi_names):
            ax = axes[row_idx, col_idx] if n_rois > 1 else axes[row_idx]
            roi_frames = roi_data[roi_name]
            ax.imshow(roi_frames[frame_idx])
            
            # Only show ROI name on first row
            if row_idx == 0:
                ax.set_title(roi_name, fontsize=8, fontweight='bold')
            
            ax.axis('off')
    
    plt.suptitle(
        f'Individual ROI Extractions (24x24) - Frames {start_frame}+{frame_indices} | ' \
        f'{META_DATA["camera_type"]} Camera | Subject: {META_DATA["subject_id"]}',
        fontsize=14, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.show()
    
    # Print shape information
    print('\nROI Shape Information for Inference:')
    print('-' * 60)
    for roi_name in roi_names:
        roi_frames = roi_data[roi_name]
        print(f'  {roi_name}:')
        print(f'    Shape: {roi_frames.shape} (frames, height, width, channels)')
        print(f'    Dtype: {roi_frames.dtype}')
        print(f'    Memory: {roi_frames.nbytes / 1024:.2f} KB')
    print()
    
    # Print inference-ready tensor information
    print('Inference-Ready Tensor Information:')
    print('-' * 60)
    print(f'  Number of frames in chunk: {n_frames}')
    print(f'  ROI box size: {ROI_BOX_SIZE}')
    print(f'  Number of ROIs: {len(roi_names)}')
    print(f'  Input tensor shape per ROI: {roi_data[roi_names[0]].shape}')
    print(f'  Total ROIs tensor shape: ({n_frames}, {len(roi_names)}, {ROI_BOX_SIZE[1]}, {ROI_BOX_SIZE[0]}, 3)')
    print()
    print('Note: For model inference, you may want to:')
    print('  1. Normalize pixel values to [0, 1] or [-1, 1]')
    print('  2. Convert to grayscale if needed')
    print('  3. Stack ROIs along channel dimension')
else:
    print('No inference data available for ROI visualization')

## 8. Detailed Shape and Size Information for Inference
Display comprehensive information about the processed data for inference compatibility.

In [ ]:
if MEDIAPIPE_AVAILABLE and 'inference_data' in locals():
    print('=' * 80)
    print('DETAILED SHAPE AND SIZE INFORMATION FOR INFERENCE')
    print('=' * 80)
    
    # Extract data
    frames = inference_data['frames']
    landmarks = inference_data['landmarks']
    roi_data = inference_data['roi_data']
    metadata = inference_data['metadata']
    
    # Video information
    print('VIDEO INFORMATION:')
    print('-' * 60)
    print(f'  Original video path: {VIDEO_PATH}')
    print(f'  Chunk start frame: {inference_data["start_frame"]}')
    print(f'  Chunk end frame: {inference_data["end_frame"]}')
    print(f'  Number of frames in chunk: {len(frames)}')
    print(f'  Frame dimensions: {frames[0].shape}')
    print(f'  Total frames memory: {frames.nbytes / (1024**2):.2f} MB')
    print()
    
    # Landmarks information
    print('LANDMARKS INFORMATION:')
    print('-' * 60)
    print(f'  Landmarks shape: {landmarks.shape}')
    print(f'  Landmarks dtype: {landmarks.dtype}')
    print(f'  Number of landmarks per frame: {landmarks.shape[1]}')
    print(f'  Landmarks memory: {landmarks.nbytes / (1024**2):.2f} MB')
    print()
    
    # ROI information
    print('ROI INFORMATION:')
    print('-' * 60)
    
    roi_stats = {}
    for roi_name, roi_frames in roi_data.items():
        roi_stats[roi_name] = {
            'shape': roi_frames.shape,
            'dtype': roi_frames.dtype,
            'memory_kb': roi_frames.nbytes / 1024,
            'num_landmarks': len(ROIS[roi_name])
        }
        print(f'  {roi_name}:')
        print(f'    Shape: {roi_frames.shape}')
        print(f'    Dtype: {roi_frames.dtype}')
        print(f'    Memory: {roi_frames.nbytes / 1024:.2f} KB')
        print(f'    Landmark indices: {len(ROIS[roi_name])} points')
        print()
    
    # Inference tensor information
    print('INFERENCE TENSOR INFORMATION:')
    print('-' * 60)
    
    # Calculate stacked ROI tensor shape
    roi_names_for_inference = [name for name in ROIS.keys() if name != 'full_face']
    n_rois = len(roi_names_for_inference)
    n_frames_chunk = len(frames)
    h, w, c = roi_data[roi_names_for_inference[0]][0].shape
    
    print(f'  Recommended input shape for models:')
    print(f'    Option 1 (separate ROIs): ({n_frames_chunk}, {h}, {w}, {c}) per ROI')
    print(f'    Option 2 (stacked ROIs): ({n_frames_chunk}, {n_rois}, {h}, {w}, {c})')
    print(f'    Option 3 (flattened): ({n_frames_chunk}, {n_rois * h * w * c})')
    print()
    
    # Memory estimation
    total_roi_memory = sum(stats['memory_kb'] for stats in roi_stats.values())
    print(f'  Total ROI memory: {total_roi_memory / 1024:.2f} MB')
    print(f'  Frames memory: {frames.nbytes / (1024**2):.2f} MB')
    print(f'  Landmarks memory: {landmarks.nbytes / (1024**2):.2f} MB')
    print(f'  Total chunk memory: {(total_roi_memory + frames.nbytes + landmarks.nbytes) / (1024**2):.2f} MB')
    print()
    
    # Metadata
    print('METADATA:')
    print('-' * 60)
    for key, value in metadata.items():
        print(f'  {key}: {value}')
    print()
    
    print('=' * 80)
    print('INFERENCE READY!')
    print('=' * 80)
    print()
    print('The data is now ready for inference. You can:')
    print('  1. Pass individual ROI arrays to your model')
    print('  2. Stack all ROIs into a single tensor')
    print('  3. Normalize and preprocess as needed')
    print('  4. Use landmarks for additional features')
else:
    print('No inference data available')

## 9. Inference Helper Functions
Utility functions for preparing data for model inference.

In [ ]:
def prepare_roi_tensor_for_inference(roi_data, exclude_full_face=True):
    '''
    Prepare ROI data as a stacked tensor for inference.
    
    Args:
        roi_data: Dictionary of ROI arrays
        exclude_full_face: Whether to exclude full_face ROI
    
    Returns:
        Stacked ROI tensor with shape (n_frames, n_rois, h, w, c)
    '''
    if exclude_full_face:
        roi_names = [name for name in roi_data.keys() if name != 'full_face']
    else:
        roi_names = list(roi_data.keys())
    
    # Get first ROI to determine dimensions
    first_roi = roi_data[roi_names[0]]
    n_frames = first_roi.shape[0]
    h, w, c = first_roi.shape[1], first_roi.shape[2], first_roi.shape[3]
    n_rois = len(roi_names)
    
    # Create stacked tensor
    stacked_tensor = np.zeros((n_frames, n_rois, h, w, c), dtype=np.float32)
    
    for roi_idx, roi_name in enumerate(roi_names):
        stacked_tensor[:, roi_idx, :, :, :] = roi_data[roi_name].astype(np.float32)
    
    return stacked_tensor

def normalize_roi_tensor(tensor):
    '''Normalize ROI tensor to [0, 1] range.'''
    return tensor / 255.0

def convert_to_grayscale(tensor):
    '''Convert RGB tensor to grayscale.'''
    # tensor shape: (n_frames, n_rois, h, w, c)
    # Convert to grayscale using luminosity method
    grayscale_tensor = np.dot(tensor[..., :3], [0.2989, 0.5870, 0.1140])
    # Add back the dimension
    return grayscale_tensor[..., np.newaxis]

def get_inference_summary(inference_data):
    '''Get a summary of inference-ready data.'''
    frames = inference_data['frames']
    roi_data = inference_data['roi_data']
    landmarks = inference_data['landmarks']
    
    roi_names = [name for name in roi_data.keys() if name != 'full_face']
    n_frames = len(frames)
    h, w, c = roi_data[roi_names[0]][0].shape
    
    summary = {
        'n_frames': n_frames,
        'n_rois': len(roi_names),
        'roi_shape': (h, w, c),
        'stacked_shape': (n_frames, len(roi_names), h, w, c),
        'landmarks_shape': landmarks.shape,
        'roi_names': roi_names,
        'metadata': inference_data.get('metadata', {})
    }
    return summary

# Test the helper functions
if MEDIAPIPE_AVAILABLE and 'inference_data' in locals():
    print('Testing inference helper functions...')
    
    # Prepare stacked tensor
    stacked_rois = prepare_roi_tensor_for_inference(inference_data['roi_data'])
    print(f'Stacked ROI tensor shape: {stacked_rois.shape}')
    print(f'Stacked ROI tensor dtype: {stacked_rois.dtype}')
    
    # Normalize
    normalized_rois = normalize_roi_tensor(stacked_rois)
    print(f'Normalized ROI tensor range: [{normalized_rois.min():.4f}, {normalized_rois.max():.4f}]')
    
    # Get summary
    summary = get_inference_summary(inference_data)
    print(f'\nInference Summary:')
    print(f'  Number of frames: {summary["n_frames"]}')
    print(f'  Number of ROIs: {summary["n_rois"]}')
    print(f'  ROI shape: {summary["roi_shape"]}')
    print(f'  Stacked shape: {summary["stacked_shape"]}')
    print(f'  Landmarks shape: {summary["landmarks_shape"]}')
    print(f'  ROI names: {summary["roi_names"]}')
    
    print('Helper functions ready for inference!')
else:
    print('No inference data available for helper function testing')

## 10. Save Inference Data
Optionally save the processed inference data for later use.

In [ ]:
if MEDIAPIPE_AVAILABLE and 'inference_data' in locals():
    print('=' * 80)
    print('SAVING INFERENCE DATA')
    print('=' * 80)
    
    # Create output directory
    inference_output_path = os.path.join(OUTPUT_PATH, 'inference_showcase')
    os.makedirs(inference_output_path, exist_ok=True)
    
    # Prepare data for saving
    save_data = {}
    
    # Save ROI data
    for roi_name, roi_frames in inference_data['roi_data'].items():
        save_data[f'roi_{roi_name}'] = roi_frames
    
    # Save landmarks
    save_data['landmarks'] = inference_data['landmarks']
    
    # Save original frames (optional, can be large)
    # save_data['frames'] = inference_data['frames']
    
    # Save metadata
    for key, value in inference_data['metadata'].items():
        save_data[f'meta_{key}'] = value
    
    save_data['start_frame'] = inference_data['start_frame']
    save_data['end_frame'] = inference_data['end_frame']
    save_data['chunk_idx'] = inference_data['chunk_idx']
    
    # Create filename
    subject_id = inference_data['metadata']['subject_id']
    camera = inference_data['metadata']['camera_type']
    condition = inference_data['metadata']['condition']
    chunk_idx = inference_data['chunk_idx']
    filename = f'inference_{subject_id}_{camera}_{condition}_chunk{chunk_idx}.npz'
    filepath = os.path.join(inference_output_path, filename)
    
    # Save as NPZ
    np.savez_compressed(filepath, **save_data)
    print(f'Saved inference data to: {filepath}')
    print(f'File size: {os.path.getsize(filepath) / (1024**2):.2f} MB')
    
    # List saved keys
    print(f'\nSaved keys:')
    with np.load(filepath) as data:
        for key in data.files:
            item = data[key]
            print(f'  {key}: shape={item.shape}, dtype={item.dtype}')
    
    print('=' * 80)
    print('INFERENCE DATA SAVED SUCCESSFULLY!')
    print('=' * 80)
else:
    print('No inference data available to save')

## 11. Summary and Next Steps

### What This Notebook Demonstrates:

1. ✅ **Video Selection**: Automatically selects a video from the MCD-rPPG dataset
2. ✅ **Chunking**: Splits video into 450-frame chunks with 150-frame overlap
3. ✅ **ROI Extraction**: Extracts 9 facial regions using MediaPipe landmarks
4. ✅ **Visualization**: Shows first, middle, and last frames with ROI boxes
5. ✅ **Individual ROI Display**: Displays extracted 24x24 ROI regions
6. ✅ **Shape Information**: Provides detailed dimensions for inference
7. ✅ **Helper Functions**: Includes utilities for preparing inference tensors
8. ✅ **Data Saving**: Optionally saves processed data as NPZ files

### Key Outputs:
- ROI bounding box visualization on original frames
- Individual ROI extractions (24x24 pixels)
- Shape and size information for all data structures
- Inference-ready tensor preparation functions
- Optional NPZ file with all processed data

### Next Steps:
- Use the helper functions to prepare data for your inference model
- Modify ROI configurations as needed
- Process additional videos using the same pipeline
- Integrate with your model's inference pipeline

In [ ]:
print('=' * 80)
print('INFERENCE VISUALIZATION SHOWCASE - COMPLETE')
print('=' * 80)
print()
print('This notebook has demonstrated:')
print('  ✓ Video chunking for inference')
print('  ✓ ROI extraction with MediaPipe landmarks')
print('  ✓ Visualization of ROI bounding boxes')
print('  ✓ Individual ROI region extraction')
print('  ✓ Shape and size information for inference')
print('  ✓ Helper functions for tensor preparation')
print()
print('Configuration used:')
print(f'  Chunk size: {CHUNK_SIZE} frames')
print(f'  Overlap: {OVERLAP_SIZE} frames')
print(f'  ROI box size: {ROI_BOX_SIZE}')
print(f'  Number of ROIs: {len(ROIS)}')
print(f'  Temporal smoothing: {SMOOTHING_WINDOW} frames')
print()
print('The processed data is ready for model inference!')